## 學習目標

1. 掌握 HuggingFace `datasets` 3.x 管線：`load_dataset` → `map(batched=True)` → `DataCollatorWithPadding`（動態 padding）。
2. 理解 `Trainer` + `TrainingArguments` 的 2026 標準寫法。
3. 掌握 `evaluate` 統一評測：`compute_metrics` 回傳 precision/recall/F1/accuracy，並分析混淆矩陣。
4. 理解 `device_map='auto'` 與 `torch_dtype=torch.bfloat16` 的載入慣例。
5. 學會用 `push_to_hub` 發佈微調模型，附最小 model card。

## 前置需求

- 已完成 `01 Tokenizer` 與 `02 Model` notebook（理解 tokenizer 輸出格式）。
- GPU 環境（至少 6 GB VRAM），或在 CPU 上以小 batch 執行（速度較慢）。

## 相鄰 Notebook

- 上一章：[02 模型載入與推論](./02%20Model%20loading_demo.ipynb)
- 下一章：[04 模型量化與 PEFT](../04Fine-tuning/04%20Fine-tuning_demo.ipynb)

## 00 版本鎖定

統一鎖版可確保本 notebook 在不同環境重現相同結果。`transformers>=4.46` 穩定支援 `BitsAndBytesConfig` 量化介面，`datasets>=3.0` 引入 `train_test_split(stratify_by_column=...)`，`evaluate>=0.4` 統一評測介面。

In [ ]:
# Install pinned versions for reproducibility
# Run once; skip if already installed
%pip install -q \
    "transformers>=4.46" \
    "datasets>=3.0" \
    "evaluate>=0.4" \
    "accelerate>=1.0" \
    "safetensors>=0.4" \
    "scikit-learn>=1.4" \
    "torch>=2.4"

## 01 載入相關套件與固定隨機種子

`set_seed(42)` 統一控制 PyTorch / NumPy / Python 的亂數，確保每次 split 與訓練結果可重現。

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)
from datasets import load_dataset
import evaluate
import numpy as np

# Fix all random seeds for reproducibility
set_seed(42)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 02 載入數據

### 資料集：ChnSentiCorp_htl_all

約 7,000 筆中文酒店評論，二元情感標籤（`0` = 負評，`1` = 正評）。正負比約 5:2，輕微不平衡。

資料直接從 HuggingFace Hub 以 `load_dataset` 讀取（Arrow 列格式，記憶體映射），分割時使用 `train_test_split(stratify_by_column='label', seed=42)` 確保正負比一致。

> 為何要 `stratify_by_column`？若不做 stratify，小驗證集可能剛好全是正評，導致驗證指標失真。

In [ ]:
# Load from HuggingFace Hub — no Google Drive mount, no local hard path
# Dataset: lansinuote/ChnSentiCorp  (7000 hotel reviews, binary label)
raw_dataset = load_dataset("lansinuote/ChnSentiCorp", split="train")
print(raw_dataset)
print(raw_dataset[0])

In [ ]:
# Check class distribution before splitting
import collections

labels = raw_dataset["label"]
counts = collections.Counter(labels)
print("Label distribution:", dict(counts))
print(f"Positive ratio: {counts[1] / len(labels):.2%}")

In [ ]:
# Stratified split: 90% train, 10% validation
# stratify_by_column preserves class ratio in both splits
split = raw_dataset.train_test_split(
    test_size=0.1,
    stratify_by_column="label",
    seed=42,
)
train_dataset = split["train"]
valid_dataset = split["test"]

print(f"Train size : {len(train_dataset)}")
print(f"Valid size : {len(valid_dataset)}")

# Verify stratification
train_pos = sum(train_dataset["label"]) / len(train_dataset)
valid_pos = sum(valid_dataset["label"]) / len(valid_dataset)
print(f"Train positive ratio : {train_pos:.2%}")
print(f"Valid positive ratio : {valid_pos:.2%}")

## 03 Tokenizer 前處理

### 動態 Padding

`map()` 階段只做 truncation，不做 padding；`DataCollatorWithPadding` 在 collate 時補到 batch 內最長序列長度。這樣每個 batch 只補到 batch 內實際需要的長度，短序列 batch 的 padding token 大幅減少，訓練速度可提升 15-30%。

### 模型選擇

`hfl/rbt3` 是 3 層 BERT-like 中文模型，參數量約 38M，適合教學展示（VRAM < 4 GB）。
如果 GPU VRAM >= 8 GB，可替換為 `hfl/chinese-roberta-wwm-ext`（12 層）。

In [ ]:
# 2026 standard: load from HuggingFace Hub directly, no git lfs clone needed
MODEL_ID = "hfl/rbt3"  # ~38M params, < 4 GB VRAM for fine-tuning
# Alternative: "hfl/chinese-roberta-wwm-ext"  # ~102M params, needs >= 8 GB VRAM

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"Vocab size  : {tokenizer.vocab_size}")
print(f"Model max len: {tokenizer.model_max_length}")

In [ ]:
def tokenize_fn(examples):
    # truncation=True only; NO padding here
    # DataCollatorWithPadding will pad to batch-max length at collation time
    return tokenizer(
        examples["text"],
        max_length=128,
        truncation=True,
    )

# batched=True: process 1000 examples at once (default batch_size=1000)
# Roughly 3-5x faster than row-by-row processing
tokenized_train = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_valid = valid_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

print(tokenized_train)
print(tokenized_train[0].keys())

In [ ]:
# DataCollatorWithPadding pads each batch to its own max length
# This replaces the fixed max_length=128 padding and manual collate_fn
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Quick sanity check: inspect one batch from validation
from torch.utils.data import DataLoader

sample_loader = DataLoader(tokenized_valid, batch_size=8, collate_fn=data_collator)
batch = next(iter(sample_loader))
print("Batch keys  :", list(batch.keys()))
print("input_ids   :", batch["input_ids"].shape)   # [8, dynamic_seq_len]
print("labels      :", batch["labels"])

## 04 模型載入（2026 標準慣例）

`AutoModelForSequenceClassification.from_pretrained` 加上以下三個關鍵參數：

- `torch_dtype=torch.bfloat16`：BF16 的指數位元（8 bit）與 FP32 相同，不需 loss scaling，訓練穩定；比 FP32 省約 50% VRAM；在 NVIDIA Ampere（A100/RTX 30xx）以上有硬體加速。
- `use_safetensors=True`：Safetensors 格式是純張量序列化，不執行任何 Python 程式碼，可避免供應鏈攻擊；另外 mmap 讀取比 pickle 快 2-3 倍。
- `device_map='auto'`（使用 `pipeline` 推論時）：Accelerate 依照可用 VRAM 自動將模型層分配至 GPU；VRAM 不足時依序 offload 到 CPU RAM 再到 disk。使用 `Trainer` 訓練時不在 `from_pretrained` 設定此參數，由 Trainer/Accelerate 內部管理。

In [ ]:
# 2026 standard model loading
# num_labels=2 adds a classification head on top of the pre-trained encoder
id2label = {0: "negative", 1: "positive"}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    torch_dtype=torch.bfloat16,   # stable BF16 instead of FP32 or FP16
    use_safetensors=True,          # safe & fast tensor format
    # device_map='auto' is handled by Trainer/Accelerate internally
    # Do NOT set it here when using Trainer — Trainer manages device placement
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"dtype           : {next(model.parameters()).dtype}")
print(f"id2label        : {model.config.id2label}")

## 05 評測指標

使用 `evaluate` 套件統一管理評測，取代手刻 `acc_num / len(validset)`。

- `accuracy`：整體正確率，但對不平衡資料集可能高估。
- `f1`（macro）：正負類的 F1 各自計算再取平均，更能反映少數類效能。
- `precision` / `recall`：協助判斷模型偏向 precision 還是 recall（視業務需求取捨）。
- 混淆矩陣：找出「把負評預測為正評」vs「把正評預測為負評」哪種錯誤更多。

In [ ]:
# Load metrics via evaluate library — unified interface for all metrics
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")
prec_metric = evaluate.load("precision")
rec_metric  = evaluate.load("recall")

def compute_metrics(eval_pred):
    """Called by Trainer after each evaluation step."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    accuracy  = acc_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1        = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    precision = prec_metric.compute(predictions=preds, references=labels, average="macro")["precision"]
    recall    = rec_metric.compute(predictions=preds, references=labels, average="macro")["recall"]

    return {
        "accuracy" : accuracy,
        "f1"       : f1,
        "precision": precision,
        "recall"   : recall,
    }

## 06 訓練（Trainer + TrainingArguments）

`Trainer` 統一處理 gradient clipping、weight decay（AdamW）、warmup、BF16 混精度、自動儲存最佳模型，無需手刻 epoch/batch 迴圈。

### TrainingArguments 關鍵參數說明

| 參數 | 值 | 說明 |
|------|----|------|
| `bf16=True` | `True` | 啟用 BF16 混精度，省 VRAM、加速 |
| `optim='adamw_torch_fused'` | - | Fused AdamW 比標準 Adam 快 10-30%；AdamW 含 weight decay，收斂更正確 |
| `warmup_ratio=0.1` | 10% | 前 10% steps 線性升溫，防止學習率過大破壞預訓練權重 |
| `lr_scheduler_type='cosine'` | - | Cosine decay 比 linear 結尾更平滑，通常效果略好 |
| `max_grad_norm=1.0` | - | Gradient clipping，防止梯度爆炸 |
| `eval_strategy='steps'` | - | 每 N steps 評估一次，比 `epoch` 更細粒度 |
| `load_best_model_at_end=True` | - | 訓練結束後自動載入最佳 checkpoint |
| `save_safetensors=True` | - | 儲存用 safetensors 格式 |

In [ ]:
import os

OUTPUT_DIR = "./rbt3-hotel-sentiment"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    # Optimizer: AdamW (weight decay correct) + fused kernel (faster)
    optim="adamw_torch_fused",
    weight_decay=0.01,
    # LR schedule: warmup 10% steps then cosine decay
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    # Gradient clipping
    max_grad_norm=1.0,
    # BF16 mixed precision (requires Ampere GPU or newer)
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported() and torch.cuda.is_available(),
    # Evaluation & checkpoint strategy
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    # Serialization
    save_safetensors=True,
    # Logging
    logging_steps=50,
    report_to="none",  # disable wandb/tensorboard for classroom
    # Reproducibility
    seed=42,
    data_seed=42,
)

print(f"BF16 enabled : {training_args.bf16}")
print(f"FP16 enabled : {training_args.fp16}")
print(f"Effective batch size : {training_args.per_device_train_batch_size} × (1 GPU) = {training_args.per_device_train_batch_size}")

In [ ]:
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,          # dynamic padding
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Trainer ready. Starting training...")
train_result = trainer.train()
print(train_result.metrics)

## 07 驗證集評測與混淆矩陣

`trainer.evaluate()` 呼叫 `compute_metrics` 並回傳完整指標。
混淆矩陣幫助識別兩類錯誤：
- **False Positive**：把負評預測為正評（對旅館業者有害）
- **False Negative**：把正評預測為負評（損失潛在正面宣傳）

In [ ]:
# Evaluate on validation set
eval_results = trainer.evaluate()
print("Validation metrics:")
for k, v in eval_results.items():
    print(f"  {k:30s}: {v:.4f}" if isinstance(v, float) else f"  {k:30s}: {v}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Get predictions on validation set
pred_output = trainer.predict(tokenized_valid)
preds = np.argmax(pred_output.predictions, axis=-1)
true_labels = pred_output.label_ids

# Confusion matrix
cm = confusion_matrix(true_labels, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["negative", "positive"])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False)
ax.set_title("Validation Confusion Matrix")
plt.tight_layout()
plt.show()

# Error analysis
fn = cm[1][0]  # positive predicted as negative
fp = cm[0][1]  # negative predicted as positive
print(f"False Negatives (positive -> negative): {fn}")
print(f"False Positives (negative -> positive): {fp}")

## 08 模型推論

### 直接推論（模型物件）

2026 寫法不需要手動 `.cuda()` 搬移輸入；`Trainer` 訓練完的模型已在正確 device。
但單步推論時需要自己把 tensor 移到 `model.device`。

In [ ]:
# Direct inference with the fine-tuned model
test_sentences = [
    "這家飯店的廁所有煙味！衣櫃有蟑螂",
    "服務很好，房間乾淨，早餐豐盛，非常值得推薦",
    "飯店生蠔，臭酸",
]

model.eval()
device = next(model.parameters()).device  # get model's device without hard-coding

with torch.inference_mode():
    inputs = tokenizer(test_sentences, padding=True, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}  # move to model's device
    logits = model(**inputs).logits
    preds  = torch.argmax(logits, dim=-1)

for sent, pred in zip(test_sentences, preds):
    label = model.config.id2label[pred.item()]
    print(f"[{label:8s}] {sent}")

### Pipeline 封裝（推薦生產用法）

`pipeline` 使用 `device_map='auto'` 讓 Accelerate 自動處理 GPU/CPU/disk offload，跨環境可移植，無需指定整數 GPU index。

In [ ]:
from transformers import pipeline

# 2026: device_map='auto' replaces device=0
pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
)

for sent in test_sentences:
    result = pipe(sent)
    print(f"{result[0]['label']:8s} (score={result[0]['score']:.3f})  {sent}")

## 09 儲存模型與 Model Card

### 本地儲存

`save_safetensors=True` 已在 `TrainingArguments` 中設定，`trainer.save_model()` 會自動使用 safetensors 格式。

### 發佈到 HuggingFace Hub

微調完的模型發佈到 Hub 可讓：
1. 其他人直接 `from_pretrained(your_hub_id)` 使用。
2. 模型 card 記錄 `id2label`、語言、任務類型，供他人快速了解用途。

In [ ]:
# Save locally with safetensors format
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model saved to {OUTPUT_DIR}")
import os
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f:40s}  {size/1e6:.2f} MB")

In [ ]:
# Optional: push to HuggingFace Hub
# Uncomment and fill in your Hub repo id before running

# HUB_REPO = "your-username/rbt3-hotel-sentiment-zh"

# trainer.push_to_hub(
#     repo_id=HUB_REPO,
#     commit_message="Add fine-tuned rbt3 for hotel sentiment classification",
#     tags=["text-classification", "chinese", "sentiment", "hotel"],
#     language="zh",
#     finetuned_from=MODEL_ID,
#     tasks="text-classification",
#     dataset_tags=["lansinuote/ChnSentiCorp"],
# )
# print(f"Model pushed to: https://huggingface.co/{HUB_REPO}")

## 10 重新載入模型（驗證可重現性）

儲存後重新載入，驗證 id2label 是否正確持久化，以及 safetensors 格式是否可正常讀取。

In [ ]:
# Reload model from local directory to verify persistence
reloaded_model = AutoModelForSequenceClassification.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)
reloaded_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

print(f"Reloaded id2label : {reloaded_model.config.id2label}")
print(f"Reloaded label2id : {reloaded_model.config.label2id}")

# Quick inference check
pipe2 = pipeline(
    "text-classification",
    model=reloaded_model,
    tokenizer=reloaded_tokenizer,
    device_map="auto",
)
print(pipe2("房間乾淨，服務很好！"))

## 11 小結與延伸練習

### 本 notebook 完成了什麼

- `load_dataset` + `map(batched=True)` 高效讀取與前處理資料。
- `DataCollatorWithPadding` 動態 padding，減少無效 padding token。
- `train_test_split(stratify_by_column)` 確保分割後類別比例一致。
- `Trainer` + `AdamW` + warmup + BF16 混精度完整訓練流程。
- `evaluate.load` + `compute_metrics` 統一回傳 F1/precision/recall/accuracy。
- `device_map='auto'` 跨環境裝置管理；safetensors 安全高效儲存。

### 延伸練習

1. **多類別分類**：把本 notebook 的二元分類改為多類別（如情感分為 1-5 星），`num_labels=5`，`compute_metrics` 改用 `average='weighted'`。
2. **學習率搜尋**：把 `learning_rate` 改為 `[1e-5, 2e-5, 5e-5]` 分三次訓練，比較驗證 F1。
3. **Gradient Accumulation**：若 GPU VRAM < 4 GB，把 `per_device_train_batch_size=8, gradient_accumulation_steps=4`，effective batch 仍為 32。
4. **換更大模型**：把 `MODEL_ID` 換成 `hfl/chinese-roberta-wwm-ext`（需 >= 8 GB VRAM），觀察 F1 變化。
5. **進入下一章**：[04 Fine-tuning with PEFT/LoRA](../04Fine-tuning/04%20Fine-tuning_demo.ipynb) — 在資源受限環境下用 4-bit 量化 + LoRA 微調更大的生成模型。